Imports

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import random, cca, copy, csv, sys

from classifiers import Classifiers
from params import Params
from graph import Graph
from dataGenerate import DataGenerate

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

Carregando parâmetros

In [16]:
params = Params.get()
energyInit              = params['energyInit']
nrCells                 = 5
t                       = params['t']
distance                = params['distance']
sample                  = params['sample']
liveEnergy              = params['liveEnergy']
cellRealocation         = params['cellRealocation']
testSamples             = params['testSamples']
trainSamples            = params['trainSamples']
totalSamples            = testSamples + trainSamples
database = 'jm1'

Carregando base de dados

In [17]:
def datasetSkLearn():
    ####### TEST Sample ############
    X, Y = make_classification(n_samples=totalSamples, n_classes=2, n_features=100, n_redundant=0, random_state=1)
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=testSamples)
    return X_train, X_test, Y_train, Y_test

def datasetJM1(multipleTrain=False):
    ####### SAMPLE #################
    with open('dataset/jm1.csv', newline='') as csvfile:
        spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
        jm1 = [row for nr, row in enumerate(spamreader)]
        jm1_true = [j for j in jm1 if j[21] == 'true']
        jm1_false = [j for j in jm1 if j[21] == 'false']
        random.shuffle(jm1_true)
        random.shuffle(jm1_false)
        trainPart = int(trainSamples/2)            #Test sample divided between true answers and false answers
        jm1_train = jm1_true[:trainPart]
        jm1_train = jm1_train + jm1_false[:trainPart]
        jm1_test = jm1_true[trainPart:trainSamples]
        jm1_test = jm1_test + jm1_false[trainPart:trainSamples]
        # jm1_test = jm1_true[trainPart:]
        # jm1_test = jm1_test + jm1_false[trainPart:]
        random.shuffle(jm1_train)
        random.shuffle(jm1_test)
        # jm1_test = jm1_test[:testSamples]

    Y_train = [j.pop(-1) for j in jm1_train]
    Y_train = [1 if x=='true' else 0 for x in Y_train]
    X_train = []
    for jt in jm1_train:
        X_train.append([float(j) for j in jt])

    # if multipleTrain:
    #     div = 4
    #     dataDiv = int((totalSamples - testSamples) / div)
    #     Y_train = [Y_train[x:x+dataDiv] for x in range(0, len(jm1_train), dataDiv)]
    #     X_train = [X_train[x:x+dataDiv] for x in range(0, len(jm1_train), dataDiv)]

    Y_test = [j.pop(-1) for j in jm1_test]
    Y_test = [1 if x=='true' else 0 for x in Y_test]
    X_test = []
    for jt in jm1_test:
        X_test.append([float(j) for j in jt])

    # Y_test_ca = [j.pop(-1) for j in jm1_test_ca]
    # Y_test_ca = [1 if x=='true' else 0 for x in Y_test_ca]
    # X_test_ca = []
    # for jt in jm1_test_ca:
    #     X_test_ca.append([float(j) for j in jt])

    # X_test = list(X_test_cf + X_test_ca)
    # Y_test = list(Y_test_cf + Y_test_ca)
    # divTest = int(testSamples/2)
    # rangeSampleCA  = range(divTest, testSamples)
    return X_train, X_test, Y_train, Y_test

def datasetNASA():
    with open('dataset/jm1.csv', newline='') as csvfile:
        spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
        base = [row for nr, row in enumerate(spamreader)]
        random.shuffle(base)
        base_train = base[:trainSamples]
        base_test = base[trainSamples:testSamples+trainSamples]
        random.shuffle(base_train)
        random.shuffle(base_test)

    Y_train = [j.pop(-1) for j in base_train]
    Y_train = [1 if x=='true' else 0 for x in Y_train]
    X_train = []
    for jt in base_train:
        X_train.append([float(j) for j in jt])

    Y_test = [j.pop(-1) for j in base_test]
    Y_test = [1 if x=='true' else 0 for x in Y_test]
    X_test = []
    for jt in base_test:
        X_test.append([float(j) for j in jt])

    return X_train, X_test, Y_train, Y_test

def datasetInference():
    Y_test = []
    X_test = []
    with open('dataset/jm1.csv', newline='') as csvfile:
        spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
        base = [row for nr, row in enumerate(spamreader)]
        random.shuffle(base)
        base = base[:testSamples]
        Y_test = [j.pop(-1) for j in base]
        Y_test = [1 if x=='true' else 0 for x in Y_test]
        X_test = []
        for jt in base:
            X_test.append([float(j) for j in jt])      
    return X_test, Y_test

def dataset():
    if database == 'jm1':
        return datasetJM1(False)
    else:
        X_train, X_test, Y_train, Y_test = datasetSkLearn()    
        divTest = int(testSamples/2)
        rangeSampleCA  = range(divTest, testSamples)
        X_test_cf = list(X_test[0:divTest])     #Sample test to train Celullar automata
        Y_test_cf = list(Y_test[0:divTest])     #Sample test to train Celullar automata
        X_test_ca = list(X_test[divTest:testSamples])   #Sample test to validate Celullar automata
        Y_test_ca = list(Y_test[divTest:testSamples])   #Sample test to validate Celullar automata
        return X_train, X_test, Y_train, Y_test, X_test_cf, Y_test_cf, X_test_ca, Y_test_ca, rangeSampleCA

# X_train, X_test, Y_train, Y_test = dataset()
X_train, X_test, Y_train, Y_test = datasetNASA()
X_test_inference, Y_train_inference = datasetInference()

Treinamento dos classificadores

In [18]:
def trainClassif(X_train, Y_train, X_test, Y_test, X_test_inference, Y_test_inference):
    def fit(clf, X_train, Y_train):
        if isinstance(X_train[0][0], list):
            nr = random.randint(0, 3)
            clf.fit(X_train[nr], Y_train[nr])
        else:
            clf.fit(X_train, Y_train)

    ####### CLASSIFIERS ############
    ClassifiersClass = Classifiers()
    names, classifiers = ClassifiersClass.getAll(ensembleFlag=True)

    classif = {}
    for name, clf in zip(names, classifiers):
        try:
            fit(clf, X_train, Y_train)
            c = {}
            c['name'] = name
            # c['predict_train'] = clf.predict(X_train)
            c['predict_test'] = clf.predict(X_test)
            c['predict_inference'] = clf.predict(X_test_inference)
            # c['prob'] = clf.predict_proba(X_test)
            # c['confidence'] = clf.decision_function(X_test)
            # c['confAvg'], c['confAvgWhenWrong'], c['confAvgWhenRight'] =  cca.confidenceInClassification(c['predict'], Y_test, c['confidence'])
            # c['score'] = clf.score(X_test_ca, Y_test_ca)
            c['score_test'] = clf.score(X_test, Y_test)
            c['score_train'] = clf.score(X_train, Y_train)
            c['score_inference'] = clf.score(X_test_inference, Y_test_inference)
            c['energy'] = energyInit
            print("Clasificador "+name+" score_train: "+str(c['score_train'])+" score_test: "+str(c['score_test'])+" score_inference: "+str(c['score_inference']))
            classif[name] = c
        except:
            print("Unexpected error:", sys.exc_info()[0])
        finally:
            continue

    badlyClassifies = list([c for c in classif if classif[c]['score_test'] < 0.5])
    for b in badlyClassifies:
        classif.pop(b)

    return classif

classif = trainClassif(X_train, Y_train, X_test, Y_test, X_test_inference, Y_train_inference)

Clasificador QDA score_train: 0.799 score_test: 0.797 score_inference: 0.795
Clasificador LDA score_train: 0.8093333333333333 score_test: 0.806 score_inference: 0.807
Clasificador SGD_PAC_00 score_train: 0.7846666666666666 score_test: 0.792 score_inference: 0.781
Clasificador SGD_PAC_01 score_train: 0.715 score_test: 0.736 score_inference: 0.721
Clasificador SGD_PAC_02 score_train: 0.7546666666666667 score_test: 0.762 score_inference: 0.757
Clasificador SGD_PAC_03 score_train: 0.781 score_test: 0.769 score_inference: 0.774
Clasificador SGD_PAC_04 score_train: 0.8013333333333333 score_test: 0.802 score_inference: 0.797
Clasificador SGD_PAC_05 score_train: 0.7406666666666667 score_test: 0.737 score_inference: 0.735
Clasificador SGD_PAC_06 score_train: 0.797 score_test: 0.799 score_inference: 0.795
Clasificador SGD_PAC_07 score_train: 0.624 score_test: 0.625 score_inference: 0.61
Clasificador SGD_PAC_08 score_train: 0.805 score_test: 0.802 score_inference: 0.799
Clasificador SGD_PAC_09 sc

D:\Users\User\miniconda3\lib\site-packages\sklearn\svm\_base.py:985: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn("Liblinear failed to converge, increase "


Clasificador LinearSVC score_train: 0.806 score_test: 0.8 score_inference: 0.793


D:\Users\User\miniconda3\lib\site-packages\sklearn\svm\_base.py:985: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn("Liblinear failed to converge, increase "


Clasificador LinearSVC_l2 score_train: 0.766 score_test: 0.763 score_inference: 0.761
Clasificador Decision_Tree_3 score_train: 0.8106666666666666 score_test: 0.803 score_inference: 0.806
Clasificador Decision_Tree_5 score_train: 0.8256666666666667 score_test: 0.804 score_inference: 0.809
Clasificador Random_Forest_12_100 score_train: 0.9133333333333333 score_test: 0.801 score_inference: 0.841
Clasificador Random_Forest_15_100 score_train: 0.9613333333333334 score_test: 0.802 score_inference: 0.851
Clasificador Random_Forest_5_300 score_train: 0.8333333333333334 score_test: 0.808 score_inference: 0.816
Clasificador Random_Forest_7_300 score_train: 0.8553333333333333 score_test: 0.803 score_inference: 0.821
Clasificador AdaBoost_50 score_train: 0.827 score_test: 0.801 score_inference: 0.802
Clasificador AdaBoost_100 score_train: 0.839 score_test: 0.796 score_inference: 0.803
Clasificador AdaBoost_150 score_train: 0.8433333333333334 score_test: 0.804 score_inference: 0.798
Clasificador E

In [19]:
print(len([c for c in Y_train_inference if c == 0]))
len(classif)

799


60

Monta Matriz

In [25]:
def resetEnergy(classif):
    for c in classif:
        classif[c]['energy'] = energyInit

def gerPoolClassif(classif):
   #shuffling classifiers for the pool
    poolClassif = list(classif.keys())
    random.shuffle(poolClassif)
    return poolClassif

def buildMatrix(classif, poolClassif):
    #building matrix of first celullar automata
    matrix = []
    for m in range(nrCells):
        matrix.append(cca.returnMatrixline(classif, poolClassif, nrCells))
    return matrix

resetEnergy(classif)
poolClassif = gerPoolClassif(classif)
matrix = buildMatrix(classif, poolClassif)

In [27]:
from numpy import true_divide
from graph import Graph
from dataGenerate import DataGenerate
import matplotlib.pyplot as plt
import copy

#Return a line of matrix
def returnMatrixline(clf, keys, nrCells):
    returnList = []
    for x in range(nrCells):
        # returnList.append(copy.deepcopy(clf[keys.pop(0)]))
        returnList.append(clf[keys.pop(0)])
    return returnList

#Return all neighbors of a determined cell
def returnNeighboringClassifiers(totalX, totalY, x, y, distance, neighbors):
    neighboringClassifiers = []
    for i in range(x-distance, x+distance+1):
        if i < 0: continue
        if i >= totalX: break
        for j in range(y-distance, y+distance+1):
            if j < 0: continue
            if j >= totalY: break
            if (i == x and j == y):
                continue
            else:
                if neighbors[i][j] != {}:
                    neighboringClassifiers.append(neighbors[i][j])
    return neighboringClassifiers

#Return true if most neighbors got it right. false if wrong. false if even.
def neighborsMajorityRight(neighbors, sampleIndex, answer):
    sumRight = 0
    totalNeighbors = len(neighbors)    
    for i in range(len(neighbors)):
        if ('predict_test' in neighbors[i]):          
            sampleCell = neighbors[i]['predict_test'][sampleIndex]
            if sampleCell == answer:
                sumRight += 1
        else: totalNeighbors -= 1
    if sumRight > (totalNeighbors/2):
        return True
    else: return False

#Returns true if the energy of the neighbors who got it right is greater than the energy of the neighbors who got it wrong. false if not. false if tie.
# def neighborsMajorityRight(neighbors, sampleIndex, answer):
#     sumRight = 0
#     sumWrong = 0
#     totalNeighbors = len(neighbors)    
#     for i in range(len(neighbors)):
#         if ('predict' in neighbors[i]):          
#             sampleCell = neighbors[i]['predict'][sampleIndex]
#             if sampleCell == answer:
#                 sumRight += neighbors[i]['energy']
#             else:
#                 sumWrong += neighbors[i]['energy']
#         else: totalNeighbors -= 1
#     if sumRight > sumWrong:
#         return True
#     else: return False

def neighborsMajorityEnergy(neighbors, sampleIndex):
    energyWhoVoteZero = sum([c['energy'] for c in neighbors if 'predict_test' in c and c['predict_test'][sampleIndex] == 0])
    energyWhoVoteOne = sum([c['energy'] for c in neighbors if 'predict_test' in c and c['predict_test'][sampleIndex] == 1])
    # energyWhoVoteZero = sum([c['energy']*c['prob'][sampleIndex][0] for c in neighbors if 'predict' in c and c['predict'][sampleIndex] == 0])
    # energyWhoVoteOne = sum([c['energy']*c['prob'][sampleIndex][1] for c in neighbors if 'predict' in c and c['predict'][sampleIndex] == 1])
    energyTotal = sum([c['energy'] for c in neighbors if 'predict_test' in c])
    return energyWhoVoteZero, energyWhoVoteOne, energyTotal

#Return average of neighbors's energy
def neighborsEnergyAverage(neighbors):
    energyTotal = sum([c['energy'] for c in neighbors if 'energy' in c])
    qtdNeighbors = len([c['energy'] for c in neighbors if 'energy' in c])

    if (energyTotal > 0) and (qtdNeighbors > 0):
        return energyTotal/qtdNeighbors
    else:
        return 0

def returnLowerEnergy(neighbors):
    return min([c['energy'] for c in neighbors if 'energy' in c])

#Return average energy of all matrix
def matrixEnergyAverage(matrix):
    size = len(matrix[0])
    return sum([sum([l['energy'] for l in m]) for m in matrix]) / (size * size)

#Return what classifies the majority neighbors choose.
def neighborsMajorityClassify(neighbors, sampleIndex):
    voteZero = ([c['predict_inference'][sampleIndex] for c in neighbors]).count(0)
    voteOne = ([c['predict_inference'][sampleIndex] for c in neighbors]).count(1)
    energyZero, energyOne, energyTotal = neighborsMajorityEnergy(neighbors, sampleIndex)
    if voteOne > voteZero:
        return 1
    elif voteOne < voteZero:
        return 0
    else:
        if energyZero >= energyOne:
            return 0
        else:
            return 1

#Method to subtract energy from live cell
def lostEnergyToLive(matrix, liveEnergy):
    matrixLength = len(matrix[0])
    for i in range(matrixLength):
        for j in range(matrixLength):
            if 'energy' in matrix[i][j]:
                matrix[i][j]['energy'] = round(matrix[i][j]['energy'] - liveEnergy, 2)


#Method to fill the empty spaces of matrix (from dead cells)
def collectOrRelocateDeadCells(matrix, pool=[], classifiers={}, cellRealocation=False, initEnergy=100):
    matrixLength = len(matrix[0])
    for i in range(matrixLength):
        for j in range(matrixLength):
            if ('energy' in matrix[i][j]) and (matrix[i][j]['energy'] <= 0):
                pool.append(matrix[i][j]['name'])
                if cellRealocation:
                    print("Classifier "+matrix[i][j]['name']+" died. "+pool[0]+" took the place.")
                    DataGenerate.saveDeadCell(matrix[i][j], classifiers[pool[0]], initEnergy)
                    matrix[i][j] = classifiers[pool.pop(0)]
                    matrix[i][j]['energy'] = initEnergy
                else: 
                    # print("cell is dead.")
                    matrix[i][j] = {}

#Return list of answers of matrix using weighted vote for each samples
def weightedVote(matrix, testSamples):
    answers = []
    matrixLength = len(matrix[0])
    for x in range(0, testSamples):
        voteOne = 0
        voteZero = 0
        for i in range(matrixLength):
            for j in range(matrixLength):
                if 'predict_test' in matrix[i][j]:
                    if matrix[i][j]['predict_test'][x] == 0:
                        voteZero += matrix[i][j]['energy']
                    if matrix[i][j]['predict_test'][x] == 1:
                        voteOne += matrix[i][j]['energy']
        if voteOne > voteZero:
            answers.append(1)
        elif voteOne < voteZero:
            answers.append(0)
        else: answers.append(2)
    return answers

#Return list of answers of matrix using weighted vote for each samples
def weightedVote2(matrix, rangeSampleCA):
    answers = []
    matrixLength = len(matrix[0])
    for x in rangeSampleCA:
        voteOne = 0
        voteZero = 0
        OneEnergy = 0
        ZeroEnergy = 0
        for i in range(matrixLength):
            for j in range(matrixLength):
                if 'predict_test' in matrix[i][j]:
                    if matrix[i][j]['predict_test'][x] == 0:
                        voteZero += 1
                        ZeroEnergy += matrix[i][j]['energy']
                    if matrix[i][j]['predict_test'][x] == 1:
                        voteOne += 1
                        OneEnergy += matrix[i][j]['energy']
        if voteOne > voteZero:
            answers.append(1)
        elif voteOne < voteZero:
            answers.append(0)
        else: 
            if OneEnergy > ZeroEnergy:
                answers.append(1)
            elif OneEnergy < ZeroEnergy:
                answers.append(0)
            else: answers.append(2)
    return answers

def weightedVoteforSample(matrix, sample):
    matrixLength = len(matrix[0])
    energyWhoVoteZero = 0
    energyWhoVoteOne = 0
    for i in range(matrixLength):
        energyZero = 0
        energyOne = 0
        energyZero, energyOne, energyTotal = neighborsMajorityEnergy(matrix[i], sample)
        energyWhoVoteZero += energyZero
        energyWhoVoteOne += energyOne
    if energyWhoVoteZero > energyWhoVoteOne:
        return 0 
    elif energyWhoVoteZero < energyWhoVoteOne:
        return 1
    else: return 2

def weightedVoteforSample2(matrix, sample):
    matrixLength = len(matrix[0])
    energyWhoVoteZero = 0
    energyWhoVoteOne = 0
    for i in range(matrixLength):
        energyZero = 0
        energyOne = 0
        energyZero, energyOne, energyTotal = neighborsMajorityEnergy(matrix[i], sample)
        energyWhoVoteZero += energyZero
        energyWhoVoteOne += energyOne
    if energyWhoVoteZero > energyWhoVoteOne:
        return 0 
    elif energyWhoVoteZero < energyWhoVoteOne:
        return 1
    else: return 2


#Compare list of samples with the list generated
def returnScore(samples, generatedList):
    total = len(samples)
    hitSum = 0
    for i in range(total):
        if samples[i] == generatedList[i]:
            hitSum += 1
    return hitSum/total
        
def transactionRuleA(currentEnergy, averageMatrix, x):
    return round(currentEnergy + x, 2)
    # if (currentEnergy > 1000): 
    #     sub = currentEnergy - (averageMatrix)
    #     if sub > 0:
    #         return round(currentEnergy + (sub/100), 2)
    #     else:
    #         return round(currentEnergy + x, 2)
    # else:
    #     return round(currentEnergy + x, 2)
    # return round(currentEnergy + (averageNeighbors * 0.001),1)

def transactionRuleB(currentEnergy, averageMatrix, x):
    return round(currentEnergy + x, 2)
    # if (currentEnergy > 1000): 
    #     sub = currentEnergy - (averageMatrix)
    #     if sub > 0:
    #         return round(currentEnergy + (sub/100), 2)
    #     else:
    #         return round(currentEnergy + x, 2)
    # else:
    #     return round(currentEnergy + x, 2)
    # return round(currentEnergy + (averageNeighbors * 0.005),1)

def transactionRuleC(currentEnergy, averageNeighbors, x):
    # return round(currentEnergy - x, 2)
    return round(currentEnergy - (averageNeighbors * x),2)
    # return round(currentEnergy - (currentEnergy * x),2)

def transactionRuleD(currentEnergy, averageNeighbors, x):
    # return round(currentEnergy - x, 2)
    return round(currentEnergy - (averageNeighbors * x),2)
    # return round(currentEnergy - (currentEnergy * x),2)

def restartEnergyMatrix(matrix, energy=100):
    matrixLength = len(matrix[0])
    for i in range(matrixLength):
        for j in range(matrixLength):
            matrix[i][j]['energy'] = energy

def algorithmCCA(matrix, Y_test_cf, nrCells, distance, pool, classif, params, qtdIteration=10, learning=True):
    #training iteration
    for x in range (0, qtdIteration):
        for sample in range(len(Y_test_cf)):
            #get each cells of matrix
            for i in range(nrCells):
                for j in range(nrCells):
                    neighbors = []
                    #neighbors of current cell
                    neighbors = returnNeighboringClassifiers(nrCells, nrCells, i, j, distance, matrix)
                    
                    #return of classifier of neighbors. True if majority right.
                    majorityNeighborsClassifier = neighborsMajorityRight(neighbors, sample, Y_test_cf[sample])
                    averageNeighborsEnergy = neighborsEnergyAverage(neighbors)
                    lowerNeighborsEnergy = returnLowerEnergy(neighbors)
                    averageMatrixEnergy = matrixEnergyAverage(matrix)
                    
                    #value of sample classified
                    if 'predict_test' in matrix[i][j]:
                        cellSample = matrix[i][j]['predict_test'][sample]
                        currentEnergy = copy.deepcopy(matrix[i][j]['energy'])
                        if cellSample == Y_test_cf[sample]:
                            #Classifier is right
                            if (majorityNeighborsClassifier):
                                matrix[i][j]['energy'] = transactionRuleA(currentEnergy, averageMatrixEnergy, params['TRA'])
                            else:
                                matrix[i][j]['energy'] = transactionRuleB(currentEnergy, averageMatrixEnergy, params['TRB'])
                        else:
                            #Classifier is wrong
                            if (majorityNeighborsClassifier):
                                matrix[i][j]['energy'] = transactionRuleC(currentEnergy, averageNeighborsEnergy, params['TRC'])
                            else:
                                matrix[i][j]['energy'] = transactionRuleD(currentEnergy, averageNeighborsEnergy, params['TRD'])
                    DataGenerate.saveStatus(matrix, classif, x, sample, i, j)
                    collectOrRelocateDeadCells(matrix, pool, classif, learning, averageNeighborsEnergy)
            # Graph.printMatrixInteractiveEnergy(matrix)
        print('it: '+str(x)+' '+ str(returnMatrixOfIndividualItem(matrix, 'energy')))
        # Graph.printMatrixInteractiveEnergy(matrix)
        # print("iteracao "+str(x))

def inferenceAlgorithm(matrix, nrCells, distance, params, testSamples, qtdIteration=100):
    classification = []
    # matrixSample = copy.deepcopy(matrix)
    for sample in range(0, testSamples):
        matrixSample = copy.deepcopy(matrix)
        for x in range(qtdIteration):
            for i in range(nrCells):
                for j in range(nrCells):
                    if 'predict_inference' in matrixSample[i][j]:
                        neighbors = []
                        #neighbors of current cell
                        neighbors = returnNeighboringClassifiers(nrCells, nrCells, i, j, distance, matrixSample)
                        neighborsVote = neighborsMajorityClassify(neighbors, sample)
                        averageNeighborsEnergy = neighborsEnergyAverage(neighbors)
                        averageMatrixEnergy = matrixEnergyAverage(matrix)
                        cellSample = matrixSample[i][j]['predict_test'][sample]
                        if cellSample == neighborsVote:
                            matrixSample[i][j]['energy'] = transactionRuleA(matrixSample[i][j]['energy'], averageMatrixEnergy, params['TRA'])
                        else:
                            matrixSample[i][j]['energy'] = transactionRuleD(matrixSample[i][j]['energy'], averageNeighborsEnergy, params['TRD'])
                        if matrixSample[i][j]['energy'] <= 0:
                            matrixSample[i][j] = {}
                    # collectOrRelocateDeadCells(matrixSample)
            a = 'a'
        # print("sample "+str(sample))
        classification.append(weightedVoteforSample(matrixSample, sample))
        print('sample: '+str(sample)+' '+ str(returnMatrixOfIndividualItem(matrixSample, 'energy')))
        # printMatrix(matrixSample)
    return classification


#################### UTILS ##############################

def returnMatrixOfIndividualItem(matrix, item):
    return [[l[item] if 'energy' in l else 0 for l in m] for m in matrix]

#Print matrix of energies in console
def printMatrix(matrix):
    energyMatrix = returnMatrixOfIndividualItem(matrix, 'energy')
    fig, ax = plt.subplots()
    im = ax.imshow(energyMatrix)
    matrixSize = len(matrix[0])
    for i in range(matrixSize):
        for j in range(matrixSize):
            text = ax.text(j, i, energyMatrix[i][j], ha="center", va="center", color="w")
    fig.tight_layout()
    plt.show()

def returnIndexOfDifferenceInLists(list1, list2):
    listIndex = []
    for i in range(len(list1)):
        if list1[i] != list2[i]:
            listIndex.append(i)
    return listIndex

def returnIndexOfEqualsInLists(list1, list2):
    listIndex = []
    for i in range(len(list1)):
        if list1[i] == list2[i]:
            listIndex.append(i)
    return listIndex

def confidenceInClassification(predict, answers, confidences):
    listEqual = returnIndexOfEqualsInLists(predict, answers)
    listDiff = returnIndexOfDifferenceInLists(predict, answers)

    confAvg = sum([abs(l) for l in confidences]) / len(confidences)
    confAvgWhenWrong = sum([abs(confidences[l]) for l in listDiff]) / len(listDiff)
    confAvgWhenRight = sum([abs(confidences[l]) for l in listEqual]) / len(listEqual)

    return confAvg, confAvgWhenWrong, confAvgWhenRight


In [28]:
params['TRA'] = 2
params['TRB'] = 15
params['TRC'] = 0.05
params['TRD'] = 0.025
# params['TRC'] = 20
# params['TRD'] = 3
DataGenerate(Y_test, classif)
algorithmCCA(matrix, Y_test, nrCells, distance, poolClassif, classif, params, t, True)
# answersList = cca.weightedVote(matrix, testSamples)
# score = cca.returnScore(Y_test, answersList)
# print("Maior score encontrado: " + str(max([classif[c]['score_test'] for c in classif])))
# print("Menor score encontrado: " + str(min([classif[c]['score_test'] for c in classif])))
# print(score)

Classifier SGD_PAC_11 died. SIGMOID_SVM took the place.
Classifier SGD_PAC_19 died. SGD_huber took the place.
Classifier SGD_PAC_02 died. SGD_PAC_26 took the place.
Classifier SGD_PAC_12 died. AdaBoost_100 took the place.
Classifier SGD_PAC_14 died. SGD_hinge took the place.
Classifier SGD_PAC_05 died. SGD_PAC_01 took the place.
Classifier SGD_PAC_27 died. LinearSVC_l2 took the place.
Classifier SGD_PAC_16 died. Gradient_Boosting took the place.
Classifier SGD_PAC_18 died. SGD_PAC_25 took the place.
Classifier SGD_squared_loss died. SGD_PAC_06 took the place.
Classifier AdaBoost_100 died. SGD_PAC_17 took the place.
Classifier SGD_PAC_25 died. Polynomial_SVM took the place.
Classifier SGD_huber died. SGD_PAC_09 took the place.
Classifier SGD_PAC_26 died. AdaBoost_150 took the place.
Classifier SGD_PAC_17 died. Random_Forest_7_300 took the place.
Classifier LinearSVC_l2 died. SGD_PAC_15 took the place.
Classifier Gradient_Boosting died. SGD_PAC_28 took the place.
Classifier Polynomial_SV

KeyboardInterrupt: 

In [ ]:
resetEnergy(classif)
answersListInference = cca.inferenceAlgorithm(matrix, nrCells, distance, params, testSamples, 10)
score2 = cca.returnScore(Y_train_inference, answersListInference)
print(score2)

In [ ]:
print(len([c for c in Y_train_inference if c == 1]))
print(len([c for c in answersListInference if c == 0]))
classif['LinearSVC']['score_inference']
classif['Decision_Tree_3']['score_inference']
classif['Nearest_Neighbors_3']['score_inference']
classif['AdaBoost_50']['score_inference']